In [1]:
import cvxpy as cp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lxml import etree

here i am just playing around with the spreadsheet from https://www.transtats.bts.gov/AverageFare/

In [3]:
fare_file = "../data/AverageFare_Q3_2025.xls"

parser = etree.XMLParser(recover=True)
tree = etree.parse(fare_file, parser)
root = tree.getroot()

ns = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}

worksheet = root.find("ss:Worksheet", ns)
rows = worksheet.findall(".//ss:Row", ns)

data_rows = []

for row in rows[2:]:   # skip title row + header row
    values = []
    current_col = 1

    for cell in row.findall("ss:Cell", ns):
        idx = cell.attrib.get("{urn:schemas-microsoft-com:office:spreadsheet}Index")
        if idx is not None:
            current_col = int(idx)

        data = cell.find("ss:Data", ns)
        text = data.text if data is not None else None

        values.append((current_col, text))
        current_col += 1

    row_dict = {}
    for col_idx, val in values:
        row_dict[col_idx] = val

    data_rows.append(row_dict)

fare_df = pd.DataFrame(data_rows)

fare_df = fare_df.rename(columns={
    1: "PassengerRank",
    2: "AirportCode",
    3: "AirportName",
    4: "CityName",
    5: "StateName",
    6: "AverageFare",
    7: "InflAdjAverageFare",
    8: "Passengers2024"
})

fare_df = fare_df[[
    "PassengerRank", "AirportCode", "AirportName", "CityName",
    "StateName", "AverageFare", "InflAdjAverageFare", "Passengers2024"
]].copy()

fare_df["PassengerRank"] = pd.to_numeric(fare_df["PassengerRank"], errors="coerce")
fare_df["AverageFare"] = pd.to_numeric(fare_df["AverageFare"], errors="coerce")
fare_df["InflAdjAverageFare"] = pd.to_numeric(fare_df["InflAdjAverageFare"], errors="coerce")
fare_df["Passengers2024"] = pd.to_numeric(fare_df["Passengers2024"], errors="coerce")

fare_df = fare_df.dropna(subset=["AirportCode", "AverageFare"]).reset_index(drop=True)

fare_df.head()

,PassengerRank,AirportCode,AirportName,CityName,StateName,AverageFare,InflAdjAverageFare,Passengers2024
0,1.0,LAX,Los Angeles International,Los Angeles,CA,401.386906,401.386906,1284083.0
1,2.0,ORD,Chicago O'Hare International,Chicago-O'Hare,IL,347.852650,347.852650,1089038.0
2,3.0,DEN,Denver International,Denver,CO,341.938870,341.938870,1023630.0
3,4.0,ATL,Hartsfield-Jackson Atlanta International,Atlanta,GA,410.818764,410.818764,1012618.0
4,5.0,BOS,Logan International,Boston,MA,355.813176,355.813176,986868.0
